# 📊 EDA Case Study — Smartphones

**Topic:** 09 EDA · **Level:** 🟠 Intermediate · **Type:** CASE STUDY

## 📖 EDA flow

1. **Setup** — display options, load cleaned CSV
2. **Univariate** — har col ka distribution dekho
3. **Bivariate** — price vs features
4. **Correlations** — numeric relations
5. **Missing fill** — KNN imputer
6. **Insights** — summaries

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [8]:
df = pd.read_csv('/content/smartphone_cleaned_v5.csv')

## 📖 Load & overview

```python
df = pd.read_csv('smartphone_cleaned_v5.csv')
df.shape                     # rows, cols
df.info()                    # dtypes + nulls
df.isnull().sum()            # null count per col
```

- Cleaned data (previous case study) load
- `info` + `isnull().sum()` — missing mapping
- `max_columns/max_rows` set — full view

In [10]:
df.shape

(980, 25)

In [9]:
df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster processor_brand  num_cores  processor_speed  \
0           False      snapdragon        8.0              3.2   
1           False      snapdragon        8.0              2.2   
2           False          exynos        8.0              2.4   
3           False      snapdragon        8.0              2.2   
4           False       dimensity        8.0              2.6   

   battery_capacity  fast_charging_available  fast_charging  ram_capacity  \
0            5000.0                        1          100.0          12.0   
1   

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 980 entries, 0 to 979
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   brand_name                 980 non-null    object 
 1   model                      980 non-null    object 
 2   price                      980 non-null    int64  
 3   rating                     879 non-null    float64
 4   has_5g                     980 non-null    bool   
 5   has_nfc                    980 non-null    bool   
 6   has_ir_blaster             980 non-null    bool   
 7   processor_brand            960 non-null    object 
 8   num_cores                  974 non-null    float64
 9   processor_speed            938 non-null    float64
 10  battery_capacity           969 non-null    float64
 11  fast_charging_available    980 non-null    int64  
 12  fast_charging              769 non-null    float64
 13  ram_capacity               980 non-null    float64

In [12]:
df.isnull().sum()

brand_name                     0
model                          0
price                          0
rating                       101
has_5g                         0
has_nfc                        0
has_ir_blaster                 0
processor_brand               20
num_cores                      6
processor_speed               42
battery_capacity              11
fast_charging_available        0
fast_charging                211
ram_capacity                   0
internal_memory                0
screen_size                    0
refresh_rate                   0
resolution                     0
num_rear_cameras               0
num_front_cameras              4
os                            14
primary_camera_rear            0
primary_camera_front           5
extended_memory_available      0
extended_upto                480
dtype: int64

In [13]:
df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster processor_brand  num_cores  processor_speed  \
0           False      snapdragon        8.0              3.2   
1           False      snapdragon        8.0              2.2   
2           False          exynos        8.0              2.4   
3           False      snapdragon        8.0              2.2   
4           False       dimensity        8.0              2.6   

   battery_capacity  fast_charging_available  fast_charging  ram_capacity  \
0            5000.0                        1          100.0          12.0   
1   

In [ ]:
# brand_name

## 📖 Brand distribution

```python
df['brand_name'].value_counts().head(10).plot(kind='bar')
df['brand_name'].value_counts().plot(kind='pie', autopct='%0.1f%%')
```

- `value_counts` → brand share
- Bar + pie — categorical univariate
- Brand market dominance dikhta hai
- `nunique` — models ka count

## 🔬 Deep Dive : Brand distribution — categorical EDA

```python
df['brand_name'].value_counts().head(10).plot(kind='bar')
df['brand_name'].value_counts().plot(kind='pie', autopct='%0.1f%%')
df['brand_name'].isnull().sum()    # missing check
```

- value_counts → frequency per brand
- bar head(10) — top brands by listings
- pie — proportion share of market
- missing check — brand completeness
- Identify dominant players
- Categorical EDA overview step
- decide top-N filtering later for plots
- compare listing counts vs prices higher-level
- Cross checks nunique for model versions
- market structure understanding

In [17]:
# plot a graph of top 5 brands
df['brand_name'].value_counts().head(10).plot(kind='bar')

<Figure size 432x288 with 1 Axes>

In [19]:
# pie chart
df['brand_name'].value_counts().plot(kind='pie',autopct='%0.1f%%')

<Figure size 432x288 with 1 Axes>

In [20]:
df['brand_name'].isnull().sum()

0

In [22]:
# model
df['model'].nunique()

980

In [23]:
# price
df['price'].describe()

count       980.000000
mean      32520.504082
std       39531.812669
min        3499.000000
25%       12999.000000
50%       19994.500000
75%       35491.500000
max      650000.000000
Name: price, dtype: float64

## 📖 Price distribution

```python
sns.displot(kind='hist', data=df, x='price', kde=True)
df['price'].skew()          # right-skew = long tail
sns.boxplot(df['price'])

df[df['price'] > 250000]    # outliers
```

- Hist + KDE — shape
- Positive skew — few flagship phones
- Boxplot — outliers visible
- Outlier rows check karo

## 🔬 Deep Dive : Price distribution — skew & outliers

```python
sns.displot(kind='hist', data=df, x='price', kde=True)
df['price'].skew()          # strong positive
sns.boxplot(df['price'])
df[df['price'] > 250000]    # luxury outliers
```

- hist + kde — shape check
- skew>0 — right tail few luxury phones
- box — Q1-Q3 spread, outliers hurt
- filter >250000 — inspect premium segment
- mean >> median in skewed data
- Decide: winsorize / log for modeling
- perspective: majority budget phones
- note for each col missing
- EDA pipeline per column: describe, skew, box, outliers, missing

In [24]:
sns.displot(kind='hist',data=df,x='price',kde=True)

<Figure size 360x360 with 1 Axes>

In [25]:
df['price'].skew()

6.591790999665567

In [26]:
sns.boxplot(df['price'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

In [28]:
df[df['price'] > 250000]

    brand_name                                     model   price  rating  \
288      apple             Apple iPhone 14 Pro Max (1TB)  182999    78.0   
319    samsung                   Samsung Galaxy Z Fold 4  154998     NaN   
427      vertu                     Vertu Signature Touch  650000    62.0   
458     xiaomi                       Xiaomi Mi Mix Alpha  199990     NaN   
478     huawei          Huawei Mate 50 RS Porsche Design  239999    81.0   
704     huawei                          Huawei Mate Xs 2  162990    89.0   
739      apple           Apple iPhone 14 Pro Max (512GB)  169900    78.0   
756      apple             Apple iPhone 13 Pro Max (1TB)  179900    86.0   
789      apple                 Apple iPhone 14 Pro (1TB)  172999    77.0   
887     xiaomi    Xiaomi Redmi K20 Pro Signature Edition  480000    88.0   
905    samsung  Samsung Galaxy Z Fold 4 (12GB RAM + 1TB)  163980     NaN   
951     huawei          Huawei Mate 30 RS Porsche Design  214990     NaN   
954     huaw

In [29]:
df['price'].isnull().sum()

0

In [30]:
df['rating'].describe()

count    879.000000
mean      78.258248
std        7.402854
min       60.000000
25%       74.000000
50%       80.000000
75%       84.000000
max       89.000000
Name: rating, dtype: float64

## 📖 Rating distribution

```python
df['rating'].describe()
df['rating'].isnull().sum() / 980   # null ratio
```

- Ratings 3.2-4.8 band
- Left skew perhaps — high rating common
- Null ratio compute — kaunsa feature fillnna chahiye

## 🔬 Deep Dive : Rating distribution — left skew

```python
df['rating'].describe()
sns.displot(kind='hist', data=df, x='rating', kde=True)
df['rating'].skew()               # negative (left)
sns.boxplot(df['rating'])
df['rating'].isnull().sum()/980   # missing proportion
```

- rating mostly 4.0-4.5 — overall good products
- negative skew — left tail low-rated
- box tight — concentrated values
- missing 2% — small, decide dropna/fill median
- describe — mean std quantiles
- informative: market product quality healthy
- each distribution summarized beside
- This drives feature decisions

In [31]:
sns.displot(kind='hist',data=df,x='rating',kde=True)

<Figure size 360x360 with 1 Axes>

In [32]:
df['rating'].skew()

-0.6989993034105535

In [33]:
sns.boxplot(df['rating'])

/usr/local/lib/python3.8/dist-packages/seaborn/_decorators.py:36: FutureWarning: Pass the following variable as a keyword arg: x. From version 0.12, the only valid positional argument will be `data`, and passing other arguments without an explicit keyword will result in an error or misinterpretation.
  warnings.warn(


<Figure size 432x288 with 1 Axes>

In [35]:
df['rating'].isnull().sum()/980

0.10306122448979592

In [36]:
df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster processor_brand  num_cores  processor_speed  \
0           False      snapdragon        8.0              3.2   
1           False      snapdragon        8.0              2.2   
2           False          exynos        8.0              2.4   
3           False      snapdragon        8.0              2.2   
4           False       dimensity        8.0              2.6   

   battery_capacity  fast_charging_available  fast_charging  ram_capacity  \
0            5000.0                        1          100.0          12.0   
1   

## 📖 Feature-wise pies

```python
df['has_5g'].value_counts().plot(kind='pie', autopct='%0.1f%%')
df['processor_brand'].value_counts().plot(kind='pie')
df['ram_capacity'].value_counts().plot(kind='pie')
```

- Boolean + categorical — pie plans
- Market standard (5G common? NFC?)
- Feature adoption dikhata hai
- Har col ke distribution — quick visual sweep

## 🔬 Deep Dive : Feature-wise pies — booleans & specs

```python
df['has_5g'].value_counts().plot(kind='pie', autopct='%0.1f%%')
df['has_nfc'].value_counts().plot(kind='pie', autopct='%0.1f%%')
df['has_ir_blaster'].value_counts().plot(kind='pie', autopct='%0.1f%%')
df[df['has_ir_blaster']==True]['brand_name'].value_counts()   # cross brand
df['num_cores'].value_counts().plot(kind='pie', autopct='%0.1f%%')
df['ram_capacity'].value_counts().plot(kind='pie', autopct='%0.1f%%')
```

- boolean features then pie proportion keeping
- each spec col its own pie
- cross-tab: which brands include IR blaster
- preprocessing distribution awareness
- spec adoption by brands - strategic
- count vs availability
- aggregated df + category proportions
- enables network between feature and brand
- quick trend: 5G now standard
- foolproof categorical EDA snapshots

In [38]:
# has_5g
df['has_5g'].value_counts().plot(kind='pie',autopct='%0.1f%%')

<Figure size 432x288 with 1 Axes>

In [39]:
# has_nfc
df['has_nfc'].value_counts().plot(kind='pie',autopct='%0.1f%%')

<Figure size 432x288 with 1 Axes>

In [40]:
# has_ir_blaster
df['has_ir_blaster'].value_counts().plot(kind='pie',autopct='%0.1f%%')

<Figure size 432x288 with 1 Axes>

In [41]:
df[df['has_ir_blaster'] == True]['brand_name'].value_counts()

xiaomi     109
poco        30
iqoo         6
huawei       6
vivo         4
redmi        2
honor        1
samsung      1
Name: brand_name, dtype: int64

In [43]:
df['processor_brand'].value_counts().plot(kind='pie',autopct="%0.1f%%")

<Figure size 432x288 with 1 Axes>

In [44]:
df['num_cores'].value_counts().plot(kind='pie',autopct="%0.1f%%")

<Figure size 432x288 with 1 Axes>

In [45]:
	
df['fast_charging_available'].value_counts().plot(kind='pie',autopct="%0.1f%%")

<Figure size 432x288 with 1 Axes>

In [46]:
	
df['ram_capacity'].value_counts().plot(kind='pie',autopct="%0.1f%%")

<Figure size 432x288 with 1 Axes>

In [47]:
df['internal_memory'].value_counts().plot(kind='pie',autopct="%0.1f%%")

<Figure size 432x288 with 1 Axes>

In [48]:
df['refresh_rate'].value_counts().plot(kind='pie',autopct="%0.1f%%")

<Figure size 432x288 with 1 Axes>

In [49]:
df['refresh_rate'].value_counts()

60     368
120    344
90     219
144     39
165      9
240      1
Name: refresh_rate, dtype: int64

In [50]:
(df['num_rear_cameras'] + df['num_front_cameras']).value_counts().plot(kind='pie',autopct="%0.1f%%")

<Figure size 432x288 with 1 Axes>

In [51]:
df['os'].value_counts().plot(kind='pie',autopct='%0.1f%%')

<Figure size 432x288 with 1 Axes>

In [52]:
# extended_memory_available
df['extended_memory_available'].value_counts().plot(kind='pie',autopct='%0.1f%%')

<Figure size 432x288 with 1 Axes>

In [53]:
df['extended_upto'].value_counts().plot(kind='pie',autopct='%0.1f%%')

<Figure size 432x288 with 1 Axes>

## 📖 Loop plotting all numerics

```python
def plot_graphs(col):
    sns.displot(kind='hist', kde=True, data=df, x=col)

num_columns = df.select_dtypes(include=['float64','int64']).iloc[:,[3,4,6,9,13,14,16]].columns
for col in num_columns:
    plot_graphs(col)
```

- `select_dtypes(numeric)` — numeric cols
- Function + loop = multi-plot
- Har col ki distribution ek saath
- EDA quick sweep technique

## 🔬 Deep Dive : Loop-plotting all numeric cols

```python
def plot_graphs(column_name):
    sns.displot(kind='hist', kde=True, data=df, x=column_name)
num_columns = df.select_dtypes(include=['float64','int64']).iloc[:,[3,4,6,9,13,14,16]]
for col in num_columns:
    plot_graphs(col)
```

- select_dtypes — pick numeric columns automatically
- iloc subset — interesting cols only
- loop each — automatic distribution plots
- reuse single plot function — DRY
- One glance all numeric distributions
- catches hidden patterns & scale issues
- cols: price, rating, processor_speed... etc
- EDA automation pattern higher productivity
- inspect each afterward manually
- figure titles update per loop

In [65]:
def plot_graphs(column_name):

  sns.displot(kind='hist',kde=True,data=df,x=column_name,label=column_name)
  sns.catplot(kind='box',data=df,x=column_name)

In [66]:
num_columns = df.select_dtypes(include=['float64','int64']).iloc[:,[3,4,6,9,13,14,16]].columns

In [67]:
for col in num_columns:
  plot_graphs(col)

<Figure size 360x360 with 1 Axes>

<Figure size 360x360 with 1 Axes>

<Figure size 360x360 with 1 Axes>

<Figure size 360x360 with 1 Axes>

<Figure size 360x360 with 1 Axes>

...

<Figure size 360x360 with 1 Axes>

<Figure size 360x360 with 1 Axes>

<Figure size 360x360 with 1 Axes>

<Figure size 360x360 with 1 Axes>

<Figure size 360x360 with 1 Axes>

In [68]:
df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster processor_brand  num_cores  processor_speed  \
0           False      snapdragon        8.0              3.2   
1           False      snapdragon        8.0              2.2   
2           False          exynos        8.0              2.4   
3           False      snapdragon        8.0              2.2   
4           False       dimensity        8.0              2.6   

   battery_capacity  fast_charging_available  fast_charging  ram_capacity  \
0            5000.0                        1          100.0          12.0   
1   

## 📖 Brand vs price

```python
sns.barplot(data=df, x='brand_name', y='price')
x = df.groupby('brand_name').count()['model']
temp_df = df[df['brand_name'].isin(x[x > 10].index)]
```

- Bar plot — brand avg price
- Few brands → cluttered bars
- Filter `count > 10` — meaningful brands
- Rotation — legible labels

## 🔬 Deep Dive : Brand vs price — barplot & filters

```python
plt.figure(figsize=(20,10))
sns.barplot(data=df, x='brand_name', y='price')
x = df.groupby('brand_name').count()['model']
temp_df = df[df['brand_name'].isin(x[x > 10].index)]   # brands with >10 models
plt.figure(figsize=(15,8))
sns.barplot(data=temp_df, x='brand_name', y='price')
```

- barplot — avg price per brand (error bars default)
- max_width brands many tiny bars not readable
- filter: groupby count model >10 - meaningful comparison
- groupby().count()['model'] - per brand listing count
- isin() - subset relevant brands
- redesigned figsize clean bars
- mean price reveals brand tiers (Apple premium)
- error bars show variance
- decision: focus analysis brands with data
- EDA selective pruning for signal

In [72]:
plt.figure(figsize=(20,10))
sns.barplot(data=df,x='brand_name',y='price')
plt.xticks(rotation='vertical')

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]),
 <a list of 46 Text major ticklabel objects>)

<Figure size 1440x720 with 1 Axes>

In [76]:
x = df.groupby('brand_name').count()['model'] 

In [82]:
temp_df = df[df['brand_name'].isin(x[x > 10].index)]

In [84]:
plt.figure(figsize=(15,8))
sns.barplot(data=temp_df,x='brand_name',y='price')
plt.xticks(rotation='vertical')

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15]),
 <a list of 16 Text major ticklabel objects>)

<Figure size 1080x576 with 1 Axes>

In [86]:
df.head()

  brand_name                      model  price  rating  has_5g  has_nfc  \
0    oneplus              OnePlus 11 5G  54999    89.0    True     True   
1    oneplus  OnePlus Nord CE 2 Lite 5G  19989    81.0    True    False   
2    samsung      Samsung Galaxy A14 5G  16499    75.0    True    False   
3   motorola       Motorola Moto G62 5G  14999    81.0    True    False   
4     realme         Realme 10 Pro Plus  24999    82.0    True    False   

   has_ir_blaster processor_brand  num_cores  processor_speed  \
0           False      snapdragon        8.0              3.2   
1           False      snapdragon        8.0              2.2   
2           False          exynos        8.0              2.4   
3           False      snapdragon        8.0              2.2   
4           False       dimensity        8.0              2.6   

   battery_capacity  fast_charging_available  fast_charging  ram_capacity  \
0            5000.0                        1          100.0          12.0   
1   

## 📖 Bivariate — numeric relations

```python
sns.scatterplot(data=df, x='rating', y='price')
sns.barplot(data=temp_df, x='has_5g', y='price', estimator=np.median)
sns.pointplot(data=temp_df, x='has_nfc', y='price', estimator=np.median)
```

- Scatter — rating vs price relationship
- `estimator=np.median` — outliers resilient
- Feature → price uplift (median)
- Categorical vs numeric via bar/point

## 🔬 Deep Dive : Bivariate relations — specs vs price

```python
sns.scatterplot(data=df, x='rating', y='price')      # rating↔price
sns.barplot(data=temp_df, x='has_5g', y='price', estimator=np.median)
sns.pointplot(data=temp_df, x='has_nfc', y='price', estimator=np.median)
sns.barplot(data=temp_df, x='processor_brand', y='price', estimator=np.median)
pd.crosstab(df['num_cores'], df['os'])
sns.scatterplot(data=df, x='screen_size', y='price')
```

- scatter — numeric×numeric correlation view
- estimator=median — robust central price per category
- barplot/pointplot bool×price — price premium features
- crosstab cores×os — joint composition
- processor brands → price banding
- screen size scatter — size vs cost
- render filled insights: 5G adds, NFC adds...
- sanity check domain expectations
- both numeric combos & categorical×num

In [87]:
sns.scatterplot(data=df,x='rating',y='price')

<Figure size 432x288 with 1 Axes>

In [89]:
sns.barplot(data=temp_df,x='has_5g',y='price',estimator=np.median)

<Figure size 432x288 with 1 Axes>

In [91]:
sns.pointplot(data=temp_df,x='has_nfc',y='price',estimator=np.median)

<Figure size 432x288 with 1 Axes>

In [93]:
sns.barplot(data=temp_df,x='has_ir_blaster',y='price',estimator=np.median)

<Figure size 432x288 with 1 Axes>

In [95]:
sns.barplot(data=temp_df,x='processor_brand',y='price',estimator=np.median)
plt.xticks(rotation='vertical')

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12]),
 <a list of 13 Text major ticklabel objects>)

<Figure size 432x288 with 1 Axes>

In [96]:
sns.barplot(data=temp_df,x='num_cores',y='price',estimator=np.median)
plt.xticks(rotation='vertical')

(array([0, 1, 2]), <a list of 3 Text major ticklabel objects>)

<Figure size 432x288 with 1 Axes>

## 📖 Cross-tab

```python
pd.crosstab(df['num_cores'], df['os'])
```

- 2 categorical values table
- Combinations count
- Trend: cores vs os pairing

In [97]:
pd.crosstab(df['num_cores'],df['os'])

os         android  ios  other
num_cores                     
4.0             33    1      1
6.0              0   39      0
8.0            875    1     10

In [98]:
sns.scatterplot(data=df,x='processor_speed',y='price')

<Figure size 432x288 with 1 Axes>

In [99]:
sns.scatterplot(data=df,x='screen_size',y='price')

<Figure size 432x288 with 1 Axes>

## 📖 Correlations

```python
df.corr()['price']
df.corr()['rating']
```

- Price ke saath har numeric col ka correlation
- Positive → price ke saath bada
- `corr()` NaN ka issue — KNN fill ke baad

## 🔬 Deep Dive : Correlations & missing imputation

```python
df.corr()['price']     # each col corr with price
df.isnull().sum()
df.corr()['rating']

x_df = df.select_dtypes(include=['int64','float64']).drop(columns='price')
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
x_df_values = imputer.fit_transform(x_df)
x_df = pd.DataFrame(x_df_values, columns=x_df.columns)
```

- corr()['price'] — strongest price drivers (screen, storage)
- missing pattern — which cols need impute
- KNNImputer — fill missing using 5-nearest rows
- fit_transform numeric matrix
- after: corr re-evaluate — compare before/after merge
- dummy variables → correlations categorical (get_dummies drop_first)
- preprocessing: impute first, then correlate clean
- Sklearn integration in EDA
- missing-aware correlations trustable
- pipeline: corr baseline → impute → corr updated

In [103]:
df.corr()['price']

price                        1.000000
rating                       0.283504
has_5g                       0.305066
has_nfc                      0.470951
has_ir_blaster              -0.015807
num_cores                   -0.048561
processor_speed              0.474049
battery_capacity            -0.159232
fast_charging_available      0.116739
fast_charging                0.277591
ram_capacity                 0.386002
internal_memory              0.557168
screen_size                  0.113253
refresh_rate                 0.244115
num_rear_cameras             0.125330
num_front_cameras            0.115228
primary_camera_rear          0.092095
primary_camera_front         0.162995
extended_memory_available   -0.448628
extended_upto                0.091945
Name: price, dtype: float64

In [104]:
df.isnull().sum()

brand_name                     0
model                          0
price                          0
rating                       101
has_5g                         0
has_nfc                        0
has_ir_blaster                 0
processor_brand               20
num_cores                      6
processor_speed               42
battery_capacity              11
fast_charging_available        0
fast_charging                211
ram_capacity                   0
internal_memory                0
screen_size                    0
refresh_rate                   0
resolution                     0
num_rear_cameras               0
num_front_cameras              4
os                            14
primary_camera_rear            0
primary_camera_front           5
extended_memory_available      0
extended_upto                480
dtype: int64

In [105]:
df.corr()['rating']

price                        0.283504
rating                       1.000000
has_5g                       0.596087
has_nfc                      0.474754
has_ir_blaster               0.156421
num_cores                    0.199741
processor_speed              0.628446
battery_capacity            -0.015581
fast_charging_available      0.542814
fast_charging                0.527613
ram_capacity                 0.757613
internal_memory              0.481070
screen_size                  0.298272
refresh_rate                 0.610795
num_rear_cameras             0.515531
num_front_cameras            0.131480
primary_camera_rear          0.562046
primary_camera_front         0.577861
extended_memory_available   -0.415265
extended_upto                0.346761
Name: rating, dtype: float64

## 📖 KNN imputer

```python
x_df = df.select_dtypes(include=['int64','float64']).drop(columns='price')
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
x_df = imputer.fit_transform(x_df)
x_df = pd.DataFrame(x_df_values, columns=x_df.columns)
```

- `KNNImputer` — missing values neighbor average
- Numeric cols only, drop target
- Fill karke correlation dobara compare
- `.corr()` ab NaN-free

In [106]:
# knn imputer
df.shape

(980, 25)

In [110]:
x_df = df.select_dtypes(include=['int64','float64']).drop(columns='price')

In [111]:
from sklearn.impute import KNNImputer

In [112]:
imputer = KNNImputer(n_neighbors=5)

In [114]:
x_df_values = imputer.fit_transform(x_df)

In [118]:
x_df = pd.DataFrame(x_df_values,columns=x_df.columns)

In [119]:
x_df['price'] = df['price']

In [120]:
x_df.head()

   rating  num_cores  processor_speed  battery_capacity  \
0    89.0        8.0              3.2            5000.0   
1    81.0        8.0              2.2            5000.0   
2    75.0        8.0              2.4            5000.0   
3    81.0        8.0              2.2            5000.0   
4    82.0        8.0              2.6            5000.0   

   fast_charging_available  fast_charging  ram_capacity  internal_memory  \
0                      1.0          100.0          12.0            256.0   
1                      1.0           33.0           6.0            128.0   
2                      1.0           15.0           4.0             64.0   
3                      1.0           29.2           6.0            128.0   
4                      1.0           67.0           6.0            128.0   

   screen_size  refresh_rate  num_rear_cameras  num_front_cameras  \
0         6.70         120.0               3.0                1.0   
1         6.59         120.0               3.0    

In [123]:
a = x_df.corr()['price'].reset_index()

In [124]:
b = df.corr()['price'].reset_index()

In [125]:
b.merge(a,on='index')

                        index   price_x   price_y
0                       price  1.000000  1.000000
1                      rating  0.283504  0.341727
2                   num_cores -0.048561 -0.055949
3             processor_speed  0.474049  0.488426
4            battery_capacity -0.159232 -0.166257
5     fast_charging_available  0.116739  0.116739
6               fast_charging  0.277591  0.220688
7                ram_capacity  0.386002  0.386002
8             internal_memory  0.557168  0.557168
9                 screen_size  0.113253  0.113253
10               refresh_rate  0.244115  0.244115
11           num_rear_cameras  0.125330  0.125330
12          num_front_cameras  0.115228  0.115787
13        primary_camera_rear  0.092095  0.092095
14       primary_camera_front  0.162995  0.160281
15  extended_memory_available -0.448628 -0.448628
16              extended_upto  0.091945 -0.004073

## 📖 Dummy + correlation

```python
pd.get_dummies(df, columns=['brand_name','processor_brand','os'], drop_first=True).corr()['price']
```

- Categorical → binary dummies
- `drop_first` — collinearity avoid
- Har category ka price relation
- EDA → feature insight → model prep

In [132]:
pd.get_dummies(df,columns=['brand_name','processor_brand','os'],drop_first=True).corr()['price']

price                         1.000000
rating                        0.283504
has_5g                        0.305066
has_nfc                       0.470951
has_ir_blaster               -0.015807
num_cores                    -0.048561
processor_speed               0.474049
battery_capacity             -0.159232
fast_charging_available       0.116739
fast_charging                 0.277591
ram_capacity                  0.386002
internal_memory               0.557168
screen_size                   0.113253
refresh_rate                  0.244115
num_rear_cameras              0.125330
num_front_cameras             0.115228
primary_camera_rear           0.092095
primary_camera_front          0.162995
extended_memory_available    -0.448628
extended_upto                 0.091945
brand_name_asus               0.090566
brand_name_blackview         -0.019033
brand_name_blu               -0.014180
brand_name_cat               -0.014173
brand_name_cola              -0.014173
brand_name_doogee        